# Bootstrap, Random Forest y declustering
Ejemplo sintético reproducible para el capítulo 13c. Distingue incertidumbre de un estadístico, evaluación predictiva y representatividad espacial.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng = np.random.default_rng(20261020)
por = np.clip(rng.normal(.18, .035, 60), .06, .32)
B = 5000
remuestras = rng.choice(por, (B, len(por)), replace=True)
medias = remuestras.mean(1)
p90 = np.quantile(remuestras, .90, axis=1)
pd.DataFrame({'estadístico':['media','P90'], 'estimación':[por.mean(),np.quantile(por,.9)], 'SE_boot':[medias.std(ddof=1),p90.std(ddof=1)], 'IC_inf':[np.quantile(medias,.025),np.quantile(p90,.025)], 'IC_sup':[np.quantile(medias,.975),np.quantile(p90,.975)]})

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
# Pozos sintéticos: la clase depende de registros y de una estructura regional.
npozos, nint = 36, 18
xw, yw = rng.uniform(0, 4, npozos), rng.uniform(0, 3, npozos)
filas = []
for w in range(npozos):
    z = np.linspace(1500, 1620, nint)
    arena = (.7*np.sin(1.4*xw[w]) + .7*np.cos(1.2*yw[w]) - .003*(z-1560) + rng.normal(0,.85,nint)) > 0
    gr = rng.normal(np.where(arena, 75, 112), 24)
    rhob = rng.normal(np.where(arena, 2.38, 2.50), .12)
    rt = np.exp(rng.normal(np.where(arena, 3.5, 3.0), .65))
    for j in range(nint): filas.append((w,xw[w],yw[w],z[j],gr[j],rhob[j],rt[j],int(arena[j])))
d = pd.DataFrame(filas, columns=['pozo','x','y','z','GR','RHOB','RT','arena'])
# Se reservan pozos completos del sector oriental: prueba espacial exigente.
test_w = np.argsort(xw)[-8:]
test = d.pozo.isin(test_w)
X = d[['GR','RHOB','RT','z']]
rf = RandomForestClassifier(n_estimators=300, max_features='sqrt', min_samples_leaf=3, random_state=42, oob_score=True)
rf.fit(X[~test], d.loc[~test,'arena'])
pred = rf.predict(X[test])
print('OOB:', rf.oob_score_)
print(classification_report(d.loc[test,'arena'], pred, target_names=['no arena','arena']))
ConfusionMatrixDisplay.from_predictions(d.loc[test,'arena'], pred, display_labels=['no arena','arena'], cmap='Blues')
plt.show()

In [ ]:
# Pesos por celdas; repetir para varios tamaños y orígenes.
def pesos_celdas(x, y, lado, origen=(0.,0.)):
    ix = np.floor((np.asarray(x)-origen[0])/lado).astype(int)
    iy = np.floor((np.asarray(y)-origen[1])/lado).astype(int)
    claves = pd.Series(list(zip(ix,iy)))
    n_celda = claves.map(claves.value_counts()).to_numpy()
    r = 1/n_celda
    return r/r.sum()
muestra = d.groupby('pozo', as_index=False).agg(x=('x','first'), y=('y','first'), porosidad=('GR',lambda s: .30-.001*s.mean()))
resultados = []
for lado in np.linspace(.2, 2.0, 10):
    for fraccion in (0, .25, .5):
        w = pesos_celdas(muestra.x, muestra.y, lado, (fraccion*lado,fraccion*lado))
        resultados.append((lado, fraccion, np.sum(w*muestra.porosidad)))
res = pd.DataFrame(resultados, columns=['lado','origen_relativo','media_declustering'])
for o,g in res.groupby('origen_relativo'): plt.plot(g.lado,g.media_declustering,'o-',label=f'origen {o}')
plt.axhline(muestra.porosidad.mean(), color='k', ls='--', label='media aritmética')
plt.xlabel('Lado de celda'); plt.ylabel('Media ponderada'); plt.legend(); plt.show()